<a href="https://colab.research.google.com/github/Crnkcbs/money_fraud/blob/main/Money_Fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BIGQUERY'DEN DOSYA YÜKLEME

##Kütüphaneler Yüklendi

In [ ]:
from google.colab import auth
from google.colab import drive
from google.cloud import bigquery
import pandas as pd

# 1. Google Drive'ınızı Colab'e sanal bir klasör olarak bağlayın
drive.mount('/content/drive')

# 2. BigQuery'e erişebilmek için Google hesabınızla kimlik doğrulaması yapın
auth.authenticate_user()
print("Google Drive ve BigQuery bağlantısı başarıyla kuruldu!")

Mounted at /content/drive
Google Drive ve BigQuery bağlantısı başarıyla kuruldu!


In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project='data426-group3')


query = """
    SELECT * FROM `data426-group3.dbt_Crnkcbs.mart_data426_group3_customer_behavior_analysis`
"""

print("Veri BigQuery'den çekiliyor, lütfen bekleyin...")
# Veriyi çekip Pandas DataFrame'e dönüştürüyoruz
df = client.query(query).to_dataframe()

# Drive'a kaydedilecek dosya adını temiz ve .csv uzantılı yapın
drive_kayit_yolu = '/content/drive/MyDrive/mart_data426_group3_customer_behavior_analysis.csv'

df.to_csv(drive_kayit_yolu, index=False)

print(f"Veri başarıyla Google Drive'a kaydedildi! Dosya Yolu: {drive_kayit_yolu}")

Veri BigQuery'den çekiliyor, lütfen bekleyin...


In [ ]:
df.head()

,transaction_id,step,type,nameorig,amount,old_balance_orig,new_balance_orig,namedest,old_balance_dest,new_balance_dest,...,new_receiver_flag,many_to_one_receiver_flag,suspicious_cashout_flag,custom_month_id,monthly_transfer_volume,is_liquidation,transfer_drain_ratio,amount_to_monthly_volume_ratio,calculated_risk_score,risk_segment
0,TX_006168500,554,TRANSFER,C193696150,3576297.10,3576297.1,3576297.1,C484597480,0.00,0.00,...,1,0,0,1,3576297.10,1,1.0000,1.0,55,Çok Yüksek Risk - Blokaj / Alarm
1,TX_004159554,304,TRANSFER,C1447502981,7617773.89,110226.0,0.0,C1438825490,190032.29,7807806.18,...,1,0,0,1,7617773.89,0,69.1105,1.0,45,Yüksek Risk - Manuel İnceleme
2,TX_004320320,308,TRANSFER,C568040605,579284.82,0.0,0.0,C971747448,1808046.72,2387331.54,...,1,0,0,1,579284.82,0,0.0000,1.0,20,Orta Risk - Ek SMS Onayı
3,TX_004943118,350,TRANSFER,C761374426,433150.09,0.0,0.0,C1826115620,1358298.85,1791448.94,...,1,0,0,1,433150.09,0,0.0000,1.0,20,Orta Risk - Ek SMS Onayı
4,TX_004921983,350,TRANSFER,C27353258,594185.21,70319.0,0.0,C961438024,138715.80,732901.01,...,1,0,0,1,594185.21,0,8.4499,1.0,35,Yüksek Risk - Manuel İnceleme


#HİPOTEZ ANALİZLERİ

##İşlem Türü ve Dolandırıcılık İlişkisi

 İşlem türü olan type ile is_fraud durumu arasında veya oluşturulan amount_bucket ile fraud senaryoları arasında birbiriyle ilişkili olup olmadığını bilimsel olarak kanıtlamak için Ki-Kare Bağımsızlık Testi (Chi-Square Test of Independence) uygulayacağız.

**Hipotezlerin Kurulması**
$H_0$ (Sıfır Hipotezi): İşlem türü ile bir işlemin fraud olması arasında hiçbir ilişki yoktur (Bu iki değişken birbirinden tamamen bağımsızdır).

$H_1$ (Alternatif Hipotez): İşlem türü ile bir işlemin fraud olması arasında anlamlı bir ilişki vardır (Bağımlıdırlar).

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. Adım: İki değişken arasında bir çapraz tablo (Contingency Table) oluşturuyoruz

contingency_table = pd.crosstab(df['type'], df['is_fraud'])

print("--- Çapraz Tablo (Gözlenen Değerler) ---")
print(contingency_table)
print("-" * 40)

# 2. Adım: Ki-Kare Testini çalıştırıyoruz
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Ki-Kare İstatistiği (Chi2): {chi2:.4f}")
print(f"P-Değeri (p-value): {p}")
print(f"Serbestlik Derecesi (dof): {dof}")

print("-" * 40)
# 3. Adım: Sonucu yorumlama
alpha = 0.05 # Standart güven sınırı (%95 güven)

if p < alpha:
    print("Sonuç: H0 hipotezi REDDEDİLDİ!")
    print("İşlem türü (type) ile dolandırıcılık (is_fraud) arasında İSTATİSTİKSEL OLARAK ANLAMLI BİR İLİŞKİ VARDIR.")
else:
    print("Sonuç: H0 hipotezi reddedilemedi.")
    print("İşlem türü ile dolandırıcılık arasında anlamlı bir ilişki bulunamamıştır (Bağımsızdırlar).")

--- Çapraz Tablo (Gözlenen Değerler) ---
is_fraud        0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097
----------------------------------------
Ki-Kare İstatistiği (Chi2): 22082.5357
P-Değeri (p-value): 0.0
Serbestlik Derecesi (dof): 4
----------------------------------------
Sonuç: H0 hipotezi REDDEDİLDİ!
İşlem türü (type) ile dolandırıcılık (is_fraud) arasında İSTATİSTİKSEL OLARAK ANLAMLI BİR İLİŞKİ VARDIR.


**Çapraz Tablonun (Contingency Table) Kritik Analizi**

Sonuca bakıldığında dolandırıcıların davranışsal parmak izi net bir şekilde görünüyor:CASH_IN, DEBIT ve PAYMENT işlemlerinde 0 (sıfır) fraud vakası var.

Toplam 8,213 fraud vakasının tamamı sadece iki işlem türünde toplanmış: CASH_OUT (4,116 vaka) ve TRANSFER (4,097 vaka).Bu durum, dolandırıcıların parayı sistem dışına kaçırmak için sadece bu iki kanalı kullandığını matematiksel olarak kanıtlıyor.

Ki-Kare Bağımsızlık testi sonucunda  İstatistik sonucu  (Chi2 & P-Value)Ki-Kare İstatistiği (22082.5357): Bu değerin bu kadar devasa çıkması, gözlenen değerler ile "eğer her şey tesadüf olsaydı beklenen değerler" arasındaki farkın uçurum olduğunu gösterir.

Yani bu dağılım asla bir tesadüf olamaz.P-Değeri (0.0): İstatistiksel olarak p-değeri tam olarak 0 olamaz, ancak o kadar küçüktür ki (örneğin $10^{-100}$'den bile küçük) bilgisayar bunu yuvarlayarak 0.0 gösterir.

Bu, %99.999... güvenle $H_0$ hipotezinin çöktüğü anlamına gelir.

##Hesap Bakiyesinin Sıfırlanması (Kritik Davranış Analizi)

In [ ]:
import pandas as pd

# 1. Sadece fraud olan işlemleri filtreleyelim
fraud_df = df[df['is_fraud'] == 1].copy()

# 2. KRİTİK DÜZELTME: Başlangıç bakiyesi 0 olanları analiz dışı bırakıyoruz!
# Sadece içinde para olan (old_balance_orig > 0) hesapları alıyoruz.
real_balance_fraud = fraud_df[fraud_df['old_balance_orig'] > 0].copy()

# 3. Hesap tamamen sıfırlandı mı kontrolü (Artık tamamen güvenli)
real_balance_fraud['is_fully_drained'] = real_balance_fraud['amount'] == real_balance_fraud['old_balance_orig']

# 4. Yüzdelik oranları hesaplayalım
drain_counts = real_balance_fraud['is_fully_drained'].value_counts()
drain_percentages = real_balance_fraud['is_fully_drained'].value_counts(normalize=True) * 100

print(f"--- Gerçek Bakiyeli Hesaplarda ({len(real_balance_fraud)} vaka) Sıfırlama Davranışı ---")
for idx in drain_counts.index:
    durum = "Hesap Tamamen Sıfırlandı (All-in)" if idx else "Kısmi Vurgun / Bakiye Kalmış"
    print(f"{durum}: {drain_counts[idx]} vaka ({drain_percentages[idx]:.2f}%)")

print("-" * 50)

--- Gerçek Bakiyeli Hesaplarda (8172 vaka) Sıfırlama Davranışı ---
Hesap Tamamen Sıfırlandı (All-in): 8018 vaka (98.12%)
Kısmi Vurgun / Bakiye Kalmış: 154 vaka (1.88%)
--------------------------------------------------


Normal işlemlerin hesap sıfırlama oranı ile Fraud işlemlerin hesap sıfırlama oranı arasındaki fark istatistiksel olarak anlamlı mı? bunun için Z-Testi (İki Örneklem Oran Z-Testi - Two-Sample Proportion Z-Test) yapılacak.

**Z-Testi (İki Örneklem Oran Z-Testi - Two-Sample Proportion Z-Test)**

Hipotezlerin Kurulması

$H_0$ (Sıfır Hipotezi): Normal işlemlerin hesap boşaltma oranı ile fraud işlemlerin hesap boşaltma oranı arasında hiçbir fark yoktur($p_1 = p_2$).

$H_1$ (Alternatif Hipotez): Fraud işlemlerin hesap boşaltma oranı, normal işlemlerin oranından anlamlı derecede daha yüksektir ($p_1 > p_2$).

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Adım: Sadece başlangıç bakiyesi 0'dan büyük olan gerçek hesapları filtreleyelim
valid_accounts = df[df['old_balance_orig'] > 0].copy()

# 2. Adım: Grupları ayıralım (Fraud olanlar ve normal olanlar)
fraud_group = valid_accounts[valid_accounts['is_fraud'] == 1].copy()
normal_group = valid_accounts[valid_accounts['is_fraud'] == 0].copy()

# 3. Adım: Her iki grupta "Hesabı Sıfırlayan" (All-in) işlem sayılarını bulalım
fraud_group['is_drained'] = fraud_group['amount'] == fraud_group['old_balance_orig']
normal_group['is_drained'] = normal_group['amount'] == normal_group['old_balance_orig']

success_fraud = fraud_group['is_drained'].sum()    # Fraud olup hesabı sıfırlayanlar
success_normal = normal_group['is_drained'].sum()  # Normal olup hesabı sıfırlayanlar

n_fraud = len(fraud_group)    # Toplam gerçek bakiyeli fraud sayısı
n_normal = len(normal_group)  # Toplam gerçek bakiyeli normal işlem sayısı

# 4. Adım: Oranları ekrana yazdıralım
print(f"Fraud Grubu Sıfırlama Oranı : %{(success_fraud / n_fraud)*100:.2f} ({success_fraud}/{n_fraud})")
print(f"Normal Grup Sıfırlama Oranı : %{(success_normal / n_normal)*100:.2f} ({success_normal}/{n_normal})")
print("-" * 50)

# 5. Adım: İki Örneklem Oran Z-Testini Çalıştıralım (Alternative='larger' çünkü fraud'un büyük olmasını bekliyoruz)
counts = np.array([success_fraud, success_normal])
nobs = np.array([n_fraud, n_normal])

z_stat, p_value = proportions_ztest(counts, nobs, alternative='larger')

print(f"Z-İstatistiği (Z-score): {z_stat:.4f}")
print(f"P-Değeri (p-value): {p_value}")
print("-" * 50)

# 6. Adım: Sonucu Yorumlama
alpha = 0.05
if p_value < alpha:
    print("Sonuç: H0 hipotezi güçlü bir şekilde REDDEDİLDİ!")
    print("Saldırganların hesap kurutma/sıfırlama oranı, normal kullanıcıların oranından İSTATİSTİKSEL OLARAK ANLAMLI DERECEDE YÜKSEKTİR.")
else:
    print("Sonuç: H0 hipotezi reddedilemedi. İki grup arasında anlamlı bir oran farkı yoktur.")

Fraud Grubu Sıfırlama Oranı : %98.12 (8018/8172)
Normal Grup Sıfırlama Oranı : %0.00 (0/4251999)
--------------------------------------------------
Z-İstatistiği (Z-score): 2044.4406
P-Değeri (p-value): 0.0
--------------------------------------------------
Sonuç: H0 hipotezi güçlü bir şekilde REDDEDİLDİ!
Saldırganların hesap kurutma/sıfırlama oranı, normal kullanıcıların oranından İSTATİSTİKSEL OLARAK ANLAMLI DERECEDE YÜKSEKTİR.


**Çıktının Derin Analizi (Verinin Söylediği Gizli Gerçek)**

Normal Grup Sıfırlama Oranı: %0.00 (0 / 4,251,999): Devasa veri kümesinde, 4.2 milyondan fazla normal işlem içinde hesabındaki parayı kuruşu kuruşuna sıfırlayan tek bir kişi bile yok! Normal insanlar transfer yaparken arkada mutlaka üç beş kuruş bakiye bırakırlar ya da küsuratlı işlem yapmazlar.

Fraud Grubu Sıfırlama Oranı: %98.12 (8,018 / 8,172): Dolandırıcıların ise neredeyse tamamı (%98.12'si) kurbanın hesabına girdiği an içerideki paranın tamamını tek seferde çekip hesabı kurutuyor.

Z-İstatistiği (2044.44): İstatistik literatüründe Z-skorunun 2 veya 3 çıkması bile iki grubun birbirinden farklı olduğunu kanıtlamak için yeterliyken, skorun 2044 çıkması iki grubun dünyalarının tamamen ayrı olduğunu gösterir. Bu bir tesadüf olamaz.

#MACHINE LEARNING

##Hangi risk grubunda ne kadar fraud var yüzdesel dağılımı

In [ ]:
# Hangi risk puanında ne kadar fraud var yüzdesel dağılımı
print(df.groupby('is_fraud')['calculated_risk_score'].describe())

              count       mean        std   min   25%   50%   75%   max
is_fraud                                                               
0         6354407.0  22.220873   8.282824  15.0  15.0  20.0  30.0  75.0
1            8213.0  45.664191  10.465027  20.0  40.0  40.0  45.0  85.0


**Risk grubu çıkan sonuca göre**

*   Mean:Uygulanan yöntemle verileri iki ana gruba ayıran en önemli fark.Fraud normal işlemlerin 2,5 katıdır

*  Min (En düşük puan):Burada en dikkatli davranan dolandırıcı bile %20 lik bir fark oluşturmuş
*   %50 (Medyan puan):Normal işlemlerin yarısından fazlası taban puan %15 de kalırken fraud vakalrının yarısı %40 puan üzerine çıkmış
*  Max : Normal bir müşteri en fazla 50 puana çıkabilmiş (muhtemelen gece yüksek miktarlı transfer yapan güvenli kullanıcılar)







##Ana eşik değeri belirleme

    **Eşik Değerleri**
    
    calculated_risk_score >= 50 THEN 'Çok Yüksek Risk - Blokaj / Alarm'
    calculated_risk_score >= 25 THEN 'Yüksek Risk - Manuel İnceleme'
    calculated_risk_score >= 15 THEN 'Orta Risk - Ek SMS Onayı'
    'Düşük Risk - Güvenli İşlem'

**0 - 49 Puan Arası**  ŞÜPHELİ (Orta/Yüksek Risk): İşlemi durdur ve 3D Secure / SMS Onayı tetikle. (Böylece o aradaki %75'lik fraud dalgası yaklanır ama araya karışabilecek az sayıda normal
müşteriyi de mağdur etmeyip SMS ile doğrulamış oluruz.

**50 ve Üzeri Puan**  KESİN BLOKAJ (Çok Yüksek Risk): İşlemi anında iptal et ve hesabı askıya al. (Çünkü normal müşterilerin %100'ü 50 puanın altında veya tam sınırda).

#BİNARY CLASSIFICATION İLE MODEL EĞİTME

## Parametreleri Belirlemek


**Belirlenen riskli müşteri davranışları ile fraud hesaplarını yakalamak**

*   Hedef (Target) Y= İs_fraud
*   Özellikler(Features) X = 	calculated_risk_score(X1),,type_code(X2),
amount_to_monthly_volume_ratio(x3),is_liquidation(X4),many_to_one_receiver_flag(X5),is_new_receiver_for_sender(X6),new_receiver_flag(X7),high_amount_flag(X8),high_velocity_flag(X9),is_night(X10),transfer_drain_ratio(X11),suspicious_cashout_flag(X12)

*   Denetimli (supervised) Öğrenmedir.

*   Veriler yapılandırılmıştır.

*   Binary Classification (İkili Sınıflandırma).

*   LightGBM ağaç tabanlı (Tree-based) modelli






## LightGBM Modelini Eğitmek

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Özellikleri (X) ve Hedefi (Y)
features = [
    'type_code', 'amount_to_monthly_volume_ratio','transfer_drain_ratio','is_liquidation',
    'many_to_one_receiver_flag', 'suspicious_cashout_flag','is_new_receiver_for_sender', 'new_receiver_flag',
    'high_amount_flag', 'high_velocity_flag', 'is_night', 'calculated_risk_score'
]

X = df[features]
y = df['is_fraud']

# 2. Veriyi %80 Eğitim, %20 Test olarak bölüyoruz (Stratify ile fraud oranını koruyoruz)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. LightGBM Modelini Tanımlıyoruz (Dengesiz veri seti ayarı aktif)

model = lgb.LGBMClassifier(

    objective='binary',
    is_unbalance=True,  # 6.3 milyon verideki fraud azlığını dengelemek için kritik parametre!
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)

# 4. Modeli Eğitiyoruz
print("Model eğitiliyor, lütfen bekleyin...")
model.fit(X_train, y_train)
print("Eğitim tamamlandı!")

Model eğitiliyor, lütfen bekleyin...
[LightGBM] [Info] Number of positive: 6570, number of negative: 5083526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.374477 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 305
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001291 -> initscore=-6.651247
[LightGBM] [Info] Start training from score -6.651247
Eğitim tamamlandı!


##Modelin Performans Ölçümü

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Test verisi üzerinde tahminler
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 2. Sonuç Raporları (Classification Report & Confusion Matrix)
print("=== MODEL: CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("=== MODEL: CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nYeni ROC-AUC Skoru: {roc_auc_score(y_test, y_pred_proba):.4f}")

# 3. Puanlama Sütununun (Calculated Risk Score) Ölçüm
importance = pd.DataFrame({'Özellik (Feature)': features, 'Önem Derecesi (Importance)': model.feature_importances_})
importance = importance.sort_values('Önem Derecesi (Importance)', ascending=False)
print("\n=== ÖZELLİK ÖNEM DERECELERİ ===")
print(importance)

=== MODEL: CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00   1270881
         1.0       0.14      0.99      0.25      1643

    accuracy                           0.99   1272524
   macro avg       0.57      0.99      0.62   1272524
weighted avg       1.00      0.99      1.00   1272524

=== MODEL: CONFUSION MATRIX ===
[[1261209    9672]
 [     12    1631]]

Yeni ROC-AUC Skoru: 0.9940

=== ÖZELLİK ÖNEM DERECELERİ ===
                 Özellik (Feature)  Önem Derecesi (Importance)
2             transfer_drain_ratio                        1940
0                        type_code                         405
10           calculated_risk_score                         236
1   amount_to_monthly_volume_ratio                         155
7                 high_amount_flag                         102
9                         is_night                          74
4        many_to_one_receiver_flag                          54


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
# Test etmek istediğimiz eşik değer aralıkları
thresholds = np.arange(0.1, 1.0, 0.1)
results = []
for t in thresholds:
    preds = (y_pred_proba >= t).astype(int)
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results.append({"Threshold": round(t, 2), "Precision": p, "Recall": r, "F1-Score": f1})
# Sonuçları net bir tablo olarak görelim
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 Threshold  Precision   Recall  F1-Score
       0.1   0.139686 0.992088  0.244892
       0.2   0.193380 0.992088  0.323670
       0.3   0.193380 0.992088  0.323670
       0.4   0.193380 0.992088  0.323670
       0.5   0.193380 0.992088  0.323670
       0.6   0.196931 0.992088  0.328629
       0.7   0.196931 0.992088  0.328629
       0.8   0.196931 0.992088  0.328629
       0.9   0.196931 0.992088  0.328629


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

# Farklı eşik değerlerini test ediyoruz
for threshold in [0.15, 0.25, 0.50, 0.70]:
    # Eğer modelin fraud olasılığı eşik değerinden büyükse 1 (fraud) kabul et
    temp_preds = np.where(y_pred_proba >= threshold, 1, 0)

    print(f"--- Eşik Değeri (Threshold): {threshold} ---")
    print(f"Yakalama Oranı (Recall): {recall_score(y_test, temp_preds):.2f}")
    print(f"Doğruluk Oranı (Precision): {precision_score(y_test, temp_preds):.2f}\n")

--- Eşik Değeri (Threshold): 0.15 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.19

--- Eşik Değeri (Threshold): 0.25 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.19

--- Eşik Değeri (Threshold): 0.5 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.19

--- Eşik Değeri (Threshold): 0.7 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.20

